In [7]:
import torch

from transformers import AutoTokenizer

from laya.pretrained import PretrainedDecisionModel
from laya.pipeline import DecisionPipeline
from laya.calibration import TemperatureScaler

device = "cpu"

In [8]:
model_name = "bert-base-uncased"

model = PretrainedDecisionModel(
    model_name=model_name,
    decision_hidden_dim=128,
)

model.load_state_dict(
    torch.load(
        "../model/grpo_model.pt",
        map_location="cpu",
        weights_only=True,
    )
)

model = model.to(device)
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18502.92it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PretrainedDecisionModel(
  (backbone): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), e

In [9]:
calibration_examples = [
    {
        "context": "I was charged twice for my monthly subscription.",
        "question": "Which team should handle this request?",
        "options": ["Billing", "Technical support", "Fraud review"],
        "target": 0,
    },
    {
        "context": "The application crashes every time I try to open it.",
        "question": "Which team should handle this request?",
        "options": ["Sales", "Technical support", "Billing"],
        "target": 1,
    },
    {
        "context": "Someone accessed my account and made unauthorized purchases.",
        "question": "Which team should handle this request?",
        "options": ["Billing", "Account support", "Fraud review"],
        "target": 2,
    },
    {
        "context": "I would like to upgrade to your enterprise subscription.",
        "question": "Which team should handle this request?",
        "options": ["Technical support", "Sales", "Refunds"],
        "target": 1,
    },
    {
        "context": "I canceled my order and want my money returned.",
        "question": "Which team should handle this request?",
        "options": ["Refunds", "Billing", "Sales"],
        "target": 0,
    },
    {
        "context": "I forgot my password and cannot sign into my account.",
        "question": "Which team should handle this request?",
        "options": ["Fraud review", "Billing", "Account support"],
        "target": 2,
    },
    {
        "context": "My payment was declined even though my card is valid.",
        "question": "Which team should handle this request?",
        "options": ["Technical support", "Billing", "Sales"],
        "target": 1,
    },
    {
        "context": "The API returns a 500 error when submitting requests.",
        "question": "Which team should handle this request?",
        "options": ["Technical support", "Refunds", "Account support"],
        "target": 0,
    },
    {
        "context": "I noticed suspicious login attempts from an unknown country.",
        "question": "Which team should handle this request?",
        "options": ["Account support", "Fraud review", "Sales"],
        "target": 1,
    },
    {
        "context": "Can someone explain the pricing for your business plan?",
        "question": "Which team should handle this request?",
        "options": ["Billing", "Sales", "Technical support"],
        "target": 1,
    },
    {
        "context": "I returned the product last week but have not received my refund.",
        "question": "Which team should handle this request?",
        "options": ["Refunds", "Fraud review", "Billing"],
        "target": 0,
    },
    {
        "context": "I need to change the email address associated with my account.",
        "question": "Which team should handle this request?",
        "options": ["Sales", "Account support", "Technical support"],
        "target": 1,
    },
]


from laya.dataset import encode_decision_batch

tokenizer = AutoTokenizer.from_pretrained(model_name)

calibration_batch = encode_decision_batch(
    calibration_examples,
    tokenizer,
)

calibration_batch = {key: value.to(device) for key, value in calibration_batch.items()}

calibration_targets = torch.tensor(
    [example["target"] for example in calibration_examples],
    dtype=torch.long,
    device=device,
)

In [10]:
from laya.calibration import TemperatureScaler

import torch.nn.functional as F

scaler = TemperatureScaler().to(device)

optimizer = torch.optim.Adam(
    scaler.parameters(),
    lr=0.01,
)

for step in range(500):
    calibration_logits, _, _ = model(
        calibration_batch["input_ids"],
        calibration_batch["attention_mask"],
        calibration_batch["option_marker_mask"],
    )
    scaled_logits = scaler(calibration_logits)

    loss = F.cross_entropy(
        scaled_logits,
        calibration_targets,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

temperature = scaler.temperature.detach().item()

print("Fitted temperature:", temperature)

Fitted temperature: 1.3675408363342285


In [11]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

pipeline = DecisionPipeline(
    model=model,
    tokenizer=tokenizer,
    temperature=temperature,
    device=device,
)

In [12]:
examples = [
    {
        "context": ("I was charged twice for the " "same subscription this month."),
        "question": ("Which team should handle " "this customer request?"),
        "options": [
            "Billing",
            "Technical support",
            "Fraud review",
        ],
    },
    {
        "context": (
            "Someone accessed my account "
            "without permission and made "
            "several suspicious transactions."
        ),
        "question": ("Which team should handle " "this customer request?"),
        "options": [
            "Billing",
            "Account support",
            "Fraud review",
        ],
    },
]

In [13]:
results = pipeline.predict(
    examples,
    confidence_threshold=0.75,
)

for result in results:
    print(result)

{'probabilities': {'Billing': 0.02651018463075161, 'Technical support': 0.018635664135217667, 'Fraud review': 0.9548541903495789}, 'prediction': 'Fraud review', 'confidence': 0.9548541903495789, 'action': 'auto'}
{'probabilities': {'Billing': 0.0009172980207949877, 'Account support': 0.0009248770074918866, 'Fraud review': 0.998157799243927}, 'prediction': 'Fraud review', 'confidence': 0.998157799243927, 'action': 'auto'}


In [14]:
pipeline.save(
    "../model/final_model.pt",
    model_name=model_name,
    decision_hidden_dim=128,
)
loaded_pipeline = DecisionPipeline.load("../model/final_model.pt", device=device)
original = pipeline.predict_proba(examples)
restored = loaded_pipeline.predict_proba(examples)
for before, after in zip(original, restored):
    assert torch.allclose(torch.tensor(before), torch.tensor(after), atol=1e-6)
print("Save/load predictions match")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15544.30it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Save/load predictions match
